# Semi-analytical methods for Bragg fiber modes


The __Bragg fiber__ is a fiber consisting of concentric rings of dielectric materials.  Similarly to the step-index fiber, we model the refractive index of a Bragg fiber as piecewise constant. What separates the Bragg fiber from the step-index fiber is a refractive index profile that alternates between high and low refractive index outwards from the core in concentric rings.

The `fibermode` repository has three modules for computing modes of Bragg fibers: `Bragg`, `BraggScalar`, and `BraggExact`. All three modules have facilities for computing modes and propagation constants of a Bragg fiber, but each achieve this through different means:

- `BraggScalar` solves for a scalar-valued field solving a Helmholtz eigenproblem (typically only appropriate for low index contrasts) by semi-analytical means.
- `BraggExact` solves for a vector-valued field solving a Maxwell eigenproblem by semi-analytical means. 
- `Bragg` solves for modes using numerical (finite element) discretizations.
  
In this document we outline the capabilities and usage of the `BraggExact` and `BraggScalar` modules, which calculate the underlying eigenpairs semi-analytically, leaving the `Bragg` case to another notebook. Our mode finding process consists of three steps:

1) Construct the Bragg fiber geometry on which we solve,
2) Find the propagation constant $\beta$,
3) Use $\beta$ to construct the mode profile.

## Matrix methods for semi-analytical solution

As in [Docs 1.1](./1_1_stepindex_exact.ipynb), we find roots by assembling a characteristic equation. Recall that for weakly-guiding step-index fibers we formed a 2x2 system of equations, derived from enforcing the continuity of the Helmholtz solutions (Bessel functions) and its derivatives at the core-cladding interface. This time, we have many concentric rings of alternating indices, and thus many more interfacial continuity conditions to be satisfied. Our modes of interest are the ones that satisfy all of these conditions.

We can utilize the _transfer matrix method_ from [[Yeh et al.](https://opg.optica.org/josa/abstract.cfm?uri=josa-68-9-1196)] to represent the continuity condition at each interface with a matrix, and the composition of these matrices from layer to layer (from inside to outside) can be used to impose the continuity condition for the entire fiber. The determinant of this composition is what yields our new characteristic equation for the Bragg fiber, and we find the zeroes of this equation to obtain propagation constants. From these propagation constants we can construct the corresponding eigenmodes.

The key difference between the two semi-analytical classes is the choice of scalar or vector representation of the field: the vector representation comes from the Maxwell model of electromagnetic fields, while the scalar representation is used for an approximate Helmholtz model.  In `BraggExact`, the continuity conditions for the electric and magnetic fields (the Maxwell interface conditions) are used at interfaces, resulting in a 4x4 transfer matrix. In `BraggScalar`, since the field is represented by a scalar mode, the mode conditions reduce to a 2x2 system.

## Constructing a Bragg fiber instance

Any instance of the `Bragg`, `BraggExact`, or `BraggScalar` demands the same construction, in the form of four lists/arrays with the same length:

- The dimensional thickness of each layer is given by `ts` (float). The first entry is the core radius, and the entries proceed outward for each non-core material layer. These values undergo non-dimensionalization prior to mesh construction by a given `scale` parameter.

- The refractive indices of each layer are given by `ns` (float).

- The material names of each layer are given by `mats` (string). The last entry of the `Bragg` class must be `Outer` to agree with the parent class.

- The maximum finite element size on the mesh subregion representing the layer is given by `maxh` (float).

Also, there are several other single parameters:

- The characteristic length used to non-dimensionalize the mesh `scale`. The `maxh` values are scaled by the nondimensionalization, so a maxh of 0.5 for a layer would be multiplied also by the given `scale` parameter.

- The operating wavelength `wl`

- The number of mesh refinements `ref`.

- A class specific input for `BraggScalar` and `BraggExact` is `no_mesh`. If set to True, no mesh is built. This can be used to save time on computation.

## A `BraggScalar` example

In [ ]:
import numpy as np
from ngsolve.webgui import Draw
from ngsolve import CF
from fibermode.bragg import Bragg, BraggExact, BraggScalar, plotlogf
from scipy.optimize import newton

### Step 1: initializing geometry
We can begin with `BraggScalar`. These inputs are the defaults:

In [ ]:
A = BraggScalar(scale=5e-5,
                ts=[5e-5, 1e-5, 2e-5],
                ns=[1, 1.44, 1],
                mats=['air', 'glass', 'air'], 
                maxhs=[.2, .02, .08], 
                bcs=None, no_mesh=False,
                wl=1.2e-6, ref=0, curve=8)

We can view the attributes of this fiber via `__dict__`:

In [ ]:
A.__dict__

From fiber theory, we know that the real parts of guided modes are bound by the inequality $kn_{\text{clad}} < \beta < kn_{\text{core}}$. The real parts of leaky modes are just below the lower bound for our guided modes (the lowest value of $k$). 

In [ ]:
k_low = A.k0 * A.ns[0] * A.scale
k_high = A.k0 * A.ns[1] * A.scale
k_low, k_high

Other important parameters include `nu`, which is related to the azimuthal variance of the mode, and `outer`, which is the way to distinguish the behavior of the Hankel function outside the fiber. The options are denoted by h1 for decaying Hankel functions associated with guided modes, and h2 for blowing up Hankel functions associated with leaky modes. In the example below, we look for a leaky mode, so we pick h2, and we want the fundamental mode, which for scalar modes means we pick nu=0.

In [ ]:
outer = 'h2'
nu = 0

### Step 2: Finding propagation constant
We can utilize the `plotlogf` function to plot the zeroes of the determinant function, with each corresponding to a propagation constant.

In [ ]:
A.determinant?

In [ ]:
plotlogf?

We can see the large scale plot:

In [ ]:
plotlogf(A.determinant,.995*k_low,1.0001*k_low, -.1,.1, nu, outer,
         iref=100, rref=100, levels=100, figsize=(12,8))

And we can also zoom in further to the fundamental mode:

In [ ]:
plotlogf(A.determinant,.9999*k_low,1.00001*k_low, -.01,.01, nu, outer,
         iref=100, rref=100, levels=100, figsize=(12,8))

For actually obtaining the constant, we can use a Newton solver (here we use the solver from SciPy).

In [ ]:
guess = np.array(.99995*k_low)

beta1 = newton(A.determinant, guess, args=(nu, outer), tol = 1e-15)

print("Scaled beta: ", beta1, ". Residual of determinant: ", abs(A.determinant(beta1, nu, outer)))

### Step 3: assemble field
With `beta1` we can assemble the field. The output of `all_fields` is an NGSolve coefficient function, so we can visualize it with `Draw()`.

In [ ]:
U = A.all_fields(beta1, nu, outer)
Draw(100*U['U'], A.mesh)

We can also utilize matplotlib to visualize the field.

In [ ]:
FsA = A.fields_matplot(beta1, nu, outer)

`FsA` is a dictionary with two functions, one for 2d and one for 1d plots.

In [ ]:
A.plot2D_contour(FsA['Ez'], figsize=(10,10))

fig, ax = A.plot1D(FsA['Ez_rad'], double_r=True, rlist=[400,10000,400], nu=nu, maxscale=True,
                  linewidth=1.5, color='k', figsize=(10,7))

## The corresponding `BraggExact` example

While the three step process of finding modes is the same in `BraggExact`, there are more field components to visualize. The exposition for these first few steps is the same, but it is important to note that in the vector case, nu is 1 for the fundamental mode.

In [ ]:
B = BraggExact(scale=5e-5,
                ts=[5e-5, 1e-5, 2e-5],
                ns=[1, 1.44, 1],
                mats=['air', 'glass', 'air'], 
                maxhs=[.2, .015, .04], 
                bcs=None, no_mesh=False,
                wl=1.2e-6, ref=0, curve=8)

In [ ]:
k_low = B.k0 * B.ns[0] * B.scale
k_high = B.k0 * B.ns[1] * B.scale
k_low, k_high

In [ ]:
outer = 'h2'
nu = 1

In [ ]:
plotlogf(B.determinant,.9999*k_low,1.00001*k_low, -.01,.01, nu, outer,
         iref=100, rref=100, levels=100, figsize=(12,8))

In [ ]:
guess = np.array(.99995*k_low)

beta2 = newton(B.determinant, guess, args=(nu, outer), tol = 1e-15)

print("Scaled beta: ", beta2, ". Residual of determinant: ", abs(B.determinant(beta2, nu, outer)))

In [ ]:
FsB = B.all_fields(beta2, nu, outer)

FsB

As you can see, `FsB` contains all of the field components. We plot the transverse (EtV) and longitudinal (Ez) components of the electric field for demonstration. 

It is important to note that, if you want clarity in the glass region to clarify any rapid oscillations, the h-refinement will increase mesh build time and draw time.

In [ ]:
Draw(FsB['Ez'], B.mesh)

For the complex fields (like `Etv`,) we can pick the real or imaginary part to plot as a vector field. Note also the `vectors` input in `Draw()` increases the amount of vectors on the mesh to see fine behavior.

In [ ]:
Draw(FsB['Etv'].real, B.mesh, vectors={'grid_size':200})

Once again we can also utilize matplotlib, which adds keys to `fsB`.

In [ ]:
fsB = B.fields_matplot(beta2, nu, outer)

fsB.keys()

In [ ]:
B.plot2D_contour(fsB['Ez'], figsize=(10,10))

We also have the ability to do 1D plots.

In [ ]:
fig, ax = A.plot1D(fsB['Ez_rad'], double_r=True, rlist=[400,10000,400], nu=nu, maxscale=True,
                  linewidth=1.5, color='k', figsize=(10,7))

There is also a streamplot functionality.

In [ ]:
mag = lambda x,y: np.sqrt(np.abs(fsB['Ex'](x,y))**2 + np.abs(fsB['Ey'](x,y))**2)

In [ ]:
fig, ax = B.plot2D_streamlines(fsB['Ex'], fsB['Ey'], contourfunc=mag, seed_nr=[2,2, 2], seed_ntheta=16, 
                               rho_linewidth=2, broken_streamlines=True,
                               maxlength=.3, plot_seed=False);